# Licență — evaluare 7B baseline pe GPU

Rulează același set de aur (60+ cazuri) prin Qwen2.5-7B-Instruct-Q4_K_M via Ollama pe GPU T4, pentru comparație cu rulul local 3B pe M1.

**Setup necesar înainte de Run All:**
1. Accelerator: GPU T4 x2 (sau P100)
2. Internet: On
3. Dataset adăugat: `licenta-code` (montat la `/kaggle/input/licenta-code/`)

## 1. Verifică GPU

In [ ]:
!nvidia-smi | head -20

## 2. Instalează Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

## 3. Pornește serverul Ollama în background

Serverul rulează pe `localhost:11434`. Folosim `nohup` ca să nu moară când celula se termină.

In [ ]:
import subprocess, time

log = open('/tmp/ollama.log', 'w')
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=log, stderr=log,
    env={'OLLAMA_HOST': '0.0.0.0:11434', 'PATH': '/usr/local/bin:/usr/bin:/bin'},
)
time.sleep(5)
print(f'Ollama PID: {ollama_proc.pid}')
!ollama list

## 4. Pull modelul Qwen2.5-7B-Q4_K_M (~4.5 GB, 3-5 minute)

In [ ]:
!ollama pull qwen2.5:7b-instruct-q4_K_M

## 5. Pregătește spațiul de lucru

Copiem datasetul read-only `/kaggle/input/licenta-code/` într-un director writable din `/kaggle/working/`, astfel încât Chroma și HuggingFace cache să poată scrie.

In [ ]:
import shutil
from pathlib import Path

WORK = Path('/kaggle/working/licenta')
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir()

shutil.copytree('/kaggle/input/licenta-code/src', WORK / 'src')
(WORK / 'data' / 'eval').mkdir(parents=True)
shutil.copy('/kaggle/input/licenta-code/data/cezicelegea_dump.json', WORK / 'data' / 'cezicelegea_dump.json')
shutil.copy('/kaggle/input/licenta-code/data/eval/gold_set.json', WORK / 'data' / 'eval' / 'gold_set.json')

import sys
sys.path.insert(0, str(WORK / 'src'))
import os
os.chdir(WORK)
print(f'Working dir: {os.getcwd()}')
!ls -R . | head -40

## 6. Instalează dependențele Python

Pin-uim aceleași versiuni ca local pentru o comparație curată.

In [ ]:
!pip install -q \
    'chromadb>=1.5' \
    'sentence-transformers<4' \
    'transformers<5' \
    'torch<2.10' \
    'pydantic>=2.13' \
    'ollama>=0.6' \
    'requests' \
    'beautifulsoup4' \
    'lxml' \
    'fpdf2'

## 7. Construiește indexul Chroma

Acesta descarcă bge-m3 (~2.3 GB, 3-5 min) la prima rulare, apoi calculează embedding-urile pentru cele 149 de chunk-uri (sub un minut pe T4).

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')

from licenta.index import build_index
build_index()

## 8. Sanity-test: o singură întrebare prin pipeline-ul 7B

Verificăm că Ollama răspunde corect cu noul model înainte de rularea completă.

In [ ]:
from licenta.retriever import Retriever
from licenta.generator import generate

MODEL = 'qwen2.5:7b-instruct-q4_K_M'

retriever = Retriever()
hits = retriever.query('Care este vârsta minimă pentru căsătorie?', k=4)
answer = generate('Care este vârsta minimă pentru căsătorie?', hits, model=MODEL)

print('Răspuns:', answer.text)
print('Surse citate:', answer.schema.cited_source_indices)
print('Valid:', answer.valid)
print('Attempts:', answer.attempts)

## 9. Rulare completă a evaluării 7B

Pe T4 ar trebui să dureze 25-40 minute (vs ~90 minute pe M1 CPU pentru 3B).

In [ ]:
from pathlib import Path
from licenta.evaluate import run

out_dir = Path('/kaggle/working/eval_7b')
report = run(
    gold_set_path=Path('data/eval/gold_set.json'),
    output_dir=out_dir,
    model=MODEL,
    k=4,
)
print(f'\nFinal report: {report}')

## 10. Afișează raportul

In [ ]:
from IPython.display import Markdown, display
with open('/kaggle/working/eval_7b/report.md') as f:
    display(Markdown(f.read()))

## 11. Descărcare rezultate

Fișierele din `/kaggle/working/eval_7b/` apar în panoul **Output** (dreapta sus). Click pe `report.md` și `results.json` pentru a le descărca.

Pune-le local în `data/eval/runs/<timestamp>_qwen2.5_7b-instruct-q4_K_M/` ca să le poți compara direct cu rulul 3B.